# Data Pipeline Overview

This notebook prepares the book dataset, converts the GBP prices to INR using the fixed-rate baseline, and validates the results by loading the cleaned data into a normalized SQLite database.

The workflow includes:
- cleaning and preparing the dataset
- adding the INR conversion column
- creating a relational schema
- running SQL queries and comparing them to pandas equivalents

In [11]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
import time


In [12]:
BASE_URL = "https://books.toscrape.com/"

# Function to scrape individual book details

def scrape_book(book_url):

    response = requests.get(book_url, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Title
    title = soup.find("h1").text.strip()

    # Price
    price = soup.find(
        "p",
        class_="price_color"
    ).text.strip()

    # Availability
    availability = soup.find(
        "p",
        class_="availability"
    ).get_text(" ", strip=True)

    # Star rating
    rating_element = soup.find(
        "p",
        class_="star-rating"
    )

    star_rating = rating_element["class"][1]

    # Category
    breadcrumb = soup.select(
        "ul.breadcrumb li a"
    )

    category = breadcrumb[-1].text.strip()

    return {
        "title": title,
        "price": price,
        "star_rating": star_rating,
        "availability": availability,
        "category": category
    }


# Scrape first 5 catalogue pages


all_books = []

for page in range(1, 5):

    page_url = (
        f"{BASE_URL}catalogue/page-{page}.html"
    )

    print(f"Scraping page {page}...")

    response = requests.get(
        page_url
    )

    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    # Find all books on the page
    books = soup.find_all(
        "article",
        class_="product_pod"
    )

    print(f"Books found: {len(books)}")

   
    # Get individual book URLs

    for book in books:

        relative_url = book.h3.a["href"]

        book_url = urljoin(
            page_url,
            relative_url
        )

        # Scrape individual book page
        book_data = scrape_book(book_url)

        all_books.append(book_data)

        # Small delay between requests
        time.sleep(0.2)



Scraping page 1...
Books found: 20
Scraping page 2...
Books found: 20
Scraping page 3...
Books found: 20
Scraping page 4...
Books found: 20


In [13]:
# Convert to DataFrame
df = pd.DataFrame(all_books)

# Verify number of books

print("\nTotal books scraped:", len(df))

print("\nFirst 10 records:")
print(df.head(10))


Total books scraped: 80

First 10 records:
                                               title    price star_rating  \
0                               A Light in the Attic  Â£51.77       Three   
1                                 Tipping the Velvet  Â£53.74         One   
2                                         Soumission  Â£50.10         One   
3                                      Sharp Objects  Â£47.82        Four   
4              Sapiens: A Brief History of Humankind  Â£54.23        Five   
5                                    The Requiem Red  Â£22.65         One   
6  The Dirty Little Secrets of Getting Your Dream...  Â£33.34        Four   
7  The Coming Woman: A Novel Based on the Life of...  Â£17.93       Three   
8  The Boys in the Boat: Nine Americans and Their...  Â£22.60        Four   
9                                    The Black Maria  Â£52.15         One   

              availability            category  
0  In stock (22 available)              Poetry  
1  In stoc

In [14]:
# Parse price text to a numeric GBP column. If a row's price is malformed, use the
# median price for that column instead of crashing the whole pipeline.
df["price_gbp"] = (
    df["price"]
    .astype(str)
    .str.strip()
    .str.replace("Â", "", regex=False)
    .str.replace("£", "", regex=False)
    .str.replace(",", "", regex=False)
    .apply(pd.to_numeric, errors="coerce")
)
median_price = df["price_gbp"].median()
df["price_gbp"] = df["price_gbp"].fillna(median_price)


In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         80 non-null     str    
 1   price         80 non-null     str    
 2   star_rating   80 non-null     str    
 3   availability  80 non-null     str    
 4   category      80 non-null     str    
 5   price_gbp     80 non-null     float64
dtypes: float64(1), str(5)
memory usage: 3.9 KB


In [16]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

# Convert star ratings to an integer column. Any malformed value is imputed with the
# median rating because rating is ordinal numeric data and a central value is a safe fallback.
df["rating"] = df["star_rating"].map(rating_map)
median_rating = df["rating"].median()
df["rating"] = df["rating"].fillna(median_rating).astype(int)



In [17]:
# Parse availability into a boolean flag. For ambiguous or malformed text, we use False
# instead of dropping the row because the stock status is a conservative binary signal and
# a False value avoids crashing the pipeline without inventing a misleading positive status.
df["in_stock"] = (
    df["availability"]
    .fillna("")
    .astype(str)
    .str.contains("In stock", case=False, na=False)
)

# Keep only rows with usable text identity fields. These are not safely recoverable from
# malformed strings, so dropping them is safer than imputing nonsense into the dataset.
df["title"] = df["title"].fillna("").astype(str).str.strip()
df["category"] = df["category"].fillna("").astype(str).str.strip()
df = df[(df["title"] != "") & (df["category"] != "")].reset_index(drop=True)

df.head()

,title,price,star_rating,availability,category,price_gbp,rating,in_stock
0,A Light in the Attic,Â£51.77,Three,In stock (22 available),Poetry,51.77,3,True
1,Tipping the Velvet,Â£53.74,One,In stock (20 available),Historical Fiction,53.74,1,True
2,Soumission,Â£50.10,One,In stock (20 available),Fiction,50.10,1,True
3,Sharp Objects,Â£47.82,Four,In stock (20 available),Mystery,47.82,4,True
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock (20 available),History,54.23,5,True


## Convert prices to INR

Using the assignment's fixed conversion rate of 1 GBP = 105.50 INR, each book price is converted into INR and stored in a dedicated column for downstream analysis and SQL storage.

In [21]:
# Required fixed-rate baseline conversion for this assignment.
# 1 GBP = 105.50 INR (project's fixed baseline conversion rate)
INR_Rate = 105.50

df["price_inr"] = (df["price_gbp"].astype(float) * INR_Rate).round(2)

df.head()

,title,price,star_rating,availability,category,price_gbp,rating,in_stock,price_inr
0,A Light in the Attic,Â£51.77,Three,In stock (22 available),Poetry,51.77,3,True,5461.74
1,Tipping the Velvet,Â£53.74,One,In stock (20 available),Historical Fiction,53.74,1,True,5669.57
2,Soumission,Â£50.10,One,In stock (20 available),Fiction,50.10,1,True,5285.55
3,Sharp Objects,Â£47.82,Four,In stock (20 available),Mystery,47.82,4,True,5045.01
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock (20 available),History,54.23,5,True,5721.26


## Create SQLite schema and validate SQL queries

This section creates the normalized book and category tables, inserts the cleaned data, and checks that the SQL query results match the equivalent pandas operations.

The final assertion confirms that the SQL join output and direct pandas merge produce the same result.

In [22]:
import sqlite3
import pandas as pd
from pandas.testing import assert_frame_equal

# 1) Create normalized SQLite schema

conn = sqlite3.connect("books.db")
conn.execute("PRAGMA foreign_keys = ON;")

# Drop in case this cell is re-run
conn.execute("DROP TABLE IF EXISTS books")
conn.execute("DROP TABLE IF EXISTS categories")

conn.execute("""
CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT NOT NULL UNIQUE
)
""")

conn.execute("""
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
)
""")

# Build category mapping from the cleaned df
category_series = (
    df["category"]
    .fillna("")
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .sort_values()
)

category_df = pd.DataFrame({
    "category_id": range(1, len(category_series) + 1),
    "category_name": category_series.tolist()
})

category_df.to_sql("categories", conn, index=False, if_exists="append")
category_lookup = dict(zip(category_df["category_name"], category_df["category_id"]))

# Insert books data with category_id foreign key
book_rows = []
for row in df[["title", "price_gbp", "price_inr", "rating", "in_stock", "category"]].itertuples(index=False):
    title = str(row.title).strip()
    price_gbp = float(row.price_gbp)
    price_inr = float(row.price_inr)
    rating = int(row.rating)
    in_stock = int(bool(row.in_stock))
    category_name = str(row.category).strip()
    category_id = int(category_lookup.get(category_name, 0))
    book_rows.append((title, price_gbp, price_inr, rating, in_stock, category_id))

conn.executemany("""
INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id)
VALUES (?, ?, ?, ?, ?, ?)
""", book_rows)

conn.commit()

# 2) SQL queries and outputs

queries = {
    "select_where": """
        SELECT title, price_inr, rating
        FROM books
        WHERE in_stock = 1 AND rating >= 4
        ORDER BY price_inr DESC
        LIMIT 10
    """,
    "order_by": """
        SELECT title, price_inr
        FROM books
        ORDER BY price_inr DESC
        LIMIT 10
    """,
    "limit": """
        SELECT title, rating
        FROM books
        ORDER BY rating DESC, title ASC
        LIMIT 5
    """,
    "distinct": """
        SELECT DISTINCT category_id
        FROM books
        ORDER BY category_id
    """,
    "between": """
        SELECT title, price_inr, rating
        FROM books
        WHERE price_inr BETWEEN 4000 AND 6000
        ORDER BY price_inr ASC
        LIMIT 10
    """,
    "join": """
        SELECT c.category_name, b.title, b.rating
        FROM books b
        JOIN categories c ON c.category_id = b.category_id
        ORDER BY c.category_name ASC, b.rating DESC, b.title ASC
        LIMIT 10
    """
}

'''query_outputs = {}

for name, query in queries.items():
    rows = conn.execute(query).fetchall()
    query_outputs[name] = {"query": query, "rows": rows}
    print(f"\n--- {name.upper()} ---")
    print(query.strip())
    for row in rows:
        print(row)'''

#3) Read back query results to pandas DataFrames

df_select_where = pd.read_sql(queries["select_where"], conn)
df_order_by = pd.read_sql(queries["order_by"], conn)
df_limit = pd.read_sql(queries["limit"], conn)
df_distinct = pd.read_sql(queries["distinct"], conn)
df_between = pd.read_sql(queries["between"], conn)
df_join_sql = pd.read_sql(queries["join"], conn)


print("\nREAD_SQL SELECT/WHERE RESULT:")
print('*'* 80)
print(df_select_where.head(10))

print("\nREAD_SQL ORDER_BY RESULT:")
print('*'* 80)
print(df_order_by.head(10))

print("\nREAD_SQL LIMIT RESULT:")
print('*'* 80)
print(df_limit.head(10))

print("\nREAD_SQL DISTINCT RESULT:")
print('*'* 80)
print(df_distinct.head(10))

print("\nREAD_SQL BETWEEN RESULT:")
print('*'* 80)
print(df_between.head(10))

print("\nREAD_SQL JOINS RESULT:")
print('*'* 80)
print(df_join_sql.head(10))

# Reproduce the join result without SQL using pandas merge
books_df_in_memory = pd.read_sql(
    "SELECT book_id, title, price_gbp, price_inr, rating, in_stock, category_id FROM books ORDER BY book_id",
    conn
)
categories_df_in_memory = pd.read_sql(
    "SELECT category_id, category_name FROM categories ORDER BY category_id",
    conn
)

df_join_merge = (
    books_df_in_memory
    .merge(categories_df_in_memory, on="category_id", how="left")
    .loc[:, ["title", "category_name", "rating"]]
    .sort_values(["category_name", "rating", "title"], ascending=[True, False, True])
    .head(10)
    .reset_index(drop=True)
)

print("\nJOIN via SQL:")
print(df_join_sql)

print("\nJOIN via direct pd.merge:")
print(df_join_merge)

# Equivalence check
df_join_sql = df_join_sql.loc[:, ["category_name", "title", "rating"]].reset_index(drop=True)
df_join_merge = df_join_merge.loc[:, ["category_name", "title", "rating"]].reset_index(drop=True)

assert_frame_equal(df_join_sql, df_join_merge)

conn.close()


READ_SQL SELECT/WHERE RESULT:
********************************************************************************
                                               title  price_inr  rating
0       The Death of Humanity: and the Case for Life    6130.60       4
1                                The Past Never Ends    5960.75       4
2              Sapiens: A Brief History of Humankind    5721.26       5
3  Scott Pilgrim's Precious Little Life (Scott Pi...    5516.60       5
4                                Behind Closed Doors    5509.21       4
5                       We Love You, Charlie Freeman    5303.48       5
6                                      Sharp Objects    5045.01       4
7                        Private Paris (Private #10)    5022.85       5
8                                     Wall and Piece    4660.99       4
9  Unseen City: The Majesty of Pigeons, the Discr...    4660.99       4

READ_SQL ORDER_BY RESULT:
*********************************************************************